# ADVC — Kaggle Notebook (Tiny-ImageNet, T4 GPU)

**Environment:** Kaggle Notebooks · GPU T4 x2 (16 GB each) · ~4 CPU cores · ~30 GB RAM

Run cells top-to-bottom. Every phase is **resumable** — re-running skips already-completed rows, and trained compression levels can be reloaded via `--skip-training`.

### Before running — notebook Settings (right sidebar)
1. **Accelerator** → `GPU T4 x2`  (never run on CPU)
2. **Internet** → `On` (needed for `git clone`, `pip install`, HF model download)
3. **Add-ons → Secrets** → add `HF_TOKEN` = your HuggingFace token
4. **Add-ons → Datasets** → attach a Tiny-ImageNet dataset, AND (from session 2 on) your saved `advc-results` dataset so prior work is restored.

### Multi-session workflow (the 36-row matrix does NOT fit in one 12h session)
Train **one or two compression levels per session**, then save results+checkpoints to a Kaggle dataset (Cell 13). Next session, attach that dataset — Cell 5b restores it and the scripts skip finished work. Do **not** rely on `/kaggle/working` persisting across sessions; it does not.

In [ ]:
# Cell 1 — Verify GPU
import torch

print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info(0)
    print('GPU name       :', name)
    print(f'Free VRAM      : {free/1e9:.1f} GB / {total/1e9:.1f} GB total')
    if 'T4' not in name:
        print('\nWARNING: expected Tesla T4 — check Settings > Accelerator > GPU T4 x2')
    else:
        print('\nT4 confirmed. Ready to proceed.')
else:
    print('\nWARNING: no GPU. Set Settings > Accelerator > GPU T4 x2, then restart.')

In [ ]:
# Cell 2 — HF token + dependencies
import os, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as e:
    print('WARNING: could not load HF_TOKEN secret:', e)
    print('Add it in Settings > Add-ons > Secrets, or downloads may be rate-limited.')

packages = ['timm', 'torchattacks', 'bitsandbytes', 'optimum', 'pyyaml',
            'tqdm', 'accelerate', 'huggingface_hub', 'transformers>=4.44.0,<5.0']
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages,
                     capture_output=True, text=True)
if res.returncode != 0:
    print('pip error:\n', res.stderr[-2000:])
else:
    print('Installed:', ', '.join(packages))

In [ ]:
# Cell 3 — Clone / update repo, set cwd
import os, sys, subprocess

REPO_URL = 'https://github.com/Jmanav/ADVC.git'
REPO_DIR = '/kaggle/working/ADVC'

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo exists — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd is now:', os.getcwd())
subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'])
assert os.path.exists('configs/base.yaml'), 'configs/base.yaml missing — clone failed?'

In [ ]:
# Cell 4 — Extract + prepare Tiny-ImageNet
# Source may be read-only (/kaggle/input mount). Output MUST be writable
# (/kaggle/working), so --root and --out are kept separate.
import os, glob, zipfile, subprocess, sys

SRC_ROOT = None
OUT_ROOT = '/kaggle/working/tiny_if'   # WRITABLE — matches configs/base.yaml

for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True):
    if os.path.isdir(p):
        SRC_ROOT = p
        break

if SRC_ROOT is None:
    zips = (glob.glob('/kaggle/input/**/tiny-imagenet-200.zip', recursive=True)
            or glob.glob('/kaggle/input/**/*tiny*imagenet*.zip', recursive=True))
    assert zips, ('No tiny-imagenet-200 folder or zip under /kaggle/input. '
                  'Attach a Tiny-ImageNet dataset in Settings > Add-ons > Datasets.')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall('/kaggle/working/')
    SRC_ROOT = '/kaggle/working/tiny-imagenet-200'

print('Source:', SRC_ROOT, '\nOutput:', OUT_ROOT)

res = subprocess.run(
    [sys.executable, 'scripts/prepare_tiny_imagenet.py',
     '--root', SRC_ROOT, '--out', OUT_ROOT],
    capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('prepare failed:\n', res.stderr[-2000:])
else:
    for d in ['train_if', 'val_if']:
        path = os.path.join(OUT_ROOT, d)
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        print(f'  {d}: {n} class folders')
    print('Tiny-ImageNet ready.')

In [ ]:
# Cell 5 — Output directories
import os
for d in ['results', 'results/tiny-imagenet',
          'results/checkpoints/at', 'results/checkpoints/atkd', 'results/figures']:
    os.makedirs(d, exist_ok=True)
    print('Ready:', d)

## Cell 5b — Restore prior results (CSVs + checkpoints)

From session 2 onward, attach your saved `advc-results` dataset (Add-ons > Datasets). This cell auto-finds it under `/kaggle/input` and restores files to the **exact paths the scripts read**:
- CSVs → `results/tiny-imagenet/` (per-dataset subfolder — this is where `dataset_results_path` writes them)
- Checkpoints → `results/checkpoints/{at,atkd}/`

If nothing is attached it prints a clear warning and everything computes fresh.

In [ ]:
# Cell 5b — Restore prior results (CSVs + checkpoints) from a saved dataset
import os, glob, shutil

DS_NAME = 'tiny-imagenet'  # cfg['dataset']['name'] — CSVs live under results/<DS_NAME>/
os.makedirs(f'results/{DS_NAME}', exist_ok=True)
os.makedirs('results/checkpoints/at', exist_ok=True)
os.makedirs('results/checkpoints/atkd', exist_ok=True)

def _find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

restored_csv = 0
for csv_name in ['phase1_results.csv', 'phase2_at_results.csv',
                 'phase2_atkd_results.csv', 'phase3_results.csv']:
    src = _find(f'/kaggle/input/**/{csv_name}')
    if src:
        shutil.copy(src, f'results/{DS_NAME}/{csv_name}')
        print('Restored CSV        ->', csv_name)
        restored_csv += 1

restored_ck = 0
for sub in ['at', 'atkd']:
    for ckpt in glob.glob(f'/kaggle/input/**/checkpoints/{sub}/*.pt', recursive=True):
        shutil.copy(ckpt, f'results/checkpoints/{sub}/{os.path.basename(ckpt)}')
        restored_ck += 1
    n = len(os.listdir(f'results/checkpoints/{sub}'))
    print(f'checkpoints/{sub}: {n} file(s) present')

if restored_csv == 0 and restored_ck == 0:
    print('\nWARNING: nothing restored from /kaggle/input. '
          'If this is not your first session, attach your advc-results dataset — '
          'otherwise everything will recompute from scratch.')

In [ ]:
# Cell 5c — Verify resume state (run before any training cell)
# Answers "is it going to restart from the beginning?" BEFORE you spend GPU.
import os, glob, pandas as pd

DS_NAME = 'tiny-imagenet'
EPOCHS = 7  # cfg['defense']['epochs'] — --skip-training only reuses the FINAL epoch

print('=== CSV rows already recorded (these attack rows will be SKIPPED) ===')
for name in ['phase1_results.csv', 'phase2_at_results.csv',
             'phase2_atkd_results.csv', 'phase3_results.csv']:
    p = f'results/{DS_NAME}/{name}'
    if os.path.exists(p):
        df = pd.read_csv(p)
        combos = sorted(set(zip(df.get('compression', []), df.get('attack', []))))
        print(f'  {name}: {len(df)} rows -> {combos}')
    else:
        print(f'  {name}: MISSING (will compute fresh)')

print('\n=== Reusable checkpoints (epoch{:02d} = loadable via --skip-training) ==='.format(EPOCHS))
for sub, prefix in [('at', 'at'), ('atkd', 'atkd')]:
    for lvl in ['fp32', 'int8', 'int4']:
        hit = glob.glob(f'results/checkpoints/{sub}/{prefix}_{lvl}_epoch{EPOCHS:02d}*.pt')
        status = ('REUSABLE ' + os.path.basename(hit[0])) if hit else 'none -> will TRAIN'
        print(f'  {sub}/{lvl}: {status}')

In [ ]:
# Cell 6 — Smoke test
import torch
from models.loader import load_config, load_model

cfg = load_config('configs/base.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dataset in config:', cfg['dataset']['name'])

model = load_model('deit_small', 'fp32', cfg, device=device)
model.eval()
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    if hasattr(out, 'logits'):
        out = out.logits
print('Output shape:', tuple(out.shape), '(expected (2, 1000))')

del model, dummy
torch.cuda.empty_cache()
print('Smoke test passed.')

## Cell 7 — Phase 1: no-defense baseline
Sweeps fp32/int8/int4 × FGSM/PGD/Patch → `results/tiny-imagenet/phase1_results.csv`. Fully resumable (skips rows already in the CSV).

In [ ]:
# Cell 7 — Phase 1
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'experiments/eval_phase1.py', '--model', 'deit_small'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Phase 1] exit code', proc.returncode)

## Cell 8 — Phase 2a: Adversarial Training (AT)

**Auto-resume:** for each compression level, if a final-epoch checkpoint already exists (from a restored session), the cell passes `--skip-training` so it **loads the checkpoint and only evaluates** — no retraining. Otherwise it trains 7 epochs.

**Multi-session tip:** set `LEVELS` below to just `['fp32', 'int8']` in session 1 and `['int4']` in session 2 to stay under the 12h cap. Each level is ~2.5–3h.

In [ ]:
# Cell 8 — Phase 2a (AT), auto-skip already-trained levels
import subprocess, sys, os, glob

LEVELS = ['fp32', 'int8', 'int4']   # trim to fit the 12h session cap
EPOCHS = 7

for compression in LEVELS:
    ckpt = glob.glob(f'results/checkpoints/at/at_{compression}_epoch{EPOCHS:02d}*.pt')
    cmd = [sys.executable, 'experiments/eval_phase2_at.py', '--compression', compression]
    if ckpt:
        cmd.append('--skip-training')
        note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
    else:
        note = 'no checkpoint -> training 7 epochs'
    print('=' * 60)
    print(f'Phase 2a: AT — {compression}  [{note}]')
    print('=' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'[Phase 2a {compression}] exit code', proc.returncode, '\n')

## Cell 9 — Phase 2b: AT + Knowledge Distillation (AT+KD)
Same auto-skip logic as Cell 8, using `results/checkpoints/atkd/`. Adds a frozen FP32 teacher (KL supervision) → `results/tiny-imagenet/phase2_atkd_results.csv`.

In [ ]:
# Cell 9 — Phase 2b (AT+KD), auto-skip already-trained levels
import subprocess, sys, os, glob

LEVELS = ['fp32', 'int8', 'int4']   # trim to fit the 12h session cap
EPOCHS = 7

for compression in LEVELS:
    ckpt = glob.glob(f'results/checkpoints/atkd/atkd_{compression}_epoch{EPOCHS:02d}*.pt')
    cmd = [sys.executable, 'experiments/eval_phase2_atkd.py', '--compression', compression]
    if ckpt:
        cmd.append('--skip-training')
        note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
    else:
        note = 'no checkpoint -> training 7 epochs'
    print('=' * 60)
    print(f'Phase 2b: AT+KD — {compression}  [{note}]')
    print('=' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'[Phase 2b {compression}] exit code', proc.returncode, '\n')

## Cell 10 — Phase 3: combined attack vs all defenses
Combined FGSM→PGD→Patch against none/AT/AT+KD → `results/tiny-imagenet/phase3_results.csv`. Requires Phase 2 checkpoints to exist (restore them via Cell 5b first).

In [ ]:
# Cell 10 — Phase 3
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'experiments/eval_phase3.py', '--model', 'deit_small'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Phase 3] exit code', proc.returncode)

In [ ]:
# Cell 11 — Preview results (correct per-dataset path)
import pandas as pd, os

DS_NAME = 'tiny-imagenet'
report = {
    'Phase 1 — No Defense': f'results/{DS_NAME}/phase1_results.csv',
    'Phase 2a — AT':        f'results/{DS_NAME}/phase2_at_results.csv',
    'Phase 2b — AT+KD':     f'results/{DS_NAME}/phase2_atkd_results.csv',
    'Phase 3 — Combined':   f'results/{DS_NAME}/phase3_results.csv',
}
cols = ['compression', 'defense', 'attack', 'clean_acc', 'robust_acc', 'asr', 'robustness_gap']
for title, path in report.items():
    print('\n' + '=' * 60 + '\n' + title + '\n' + '=' * 60)
    if not os.path.exists(path):
        print('  not yet generated:', path); continue
    df = pd.read_csv(path)
    if df.empty:
        print('  file exists but empty'); continue
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    print(f'  {len(df)} row(s)')

In [ ]:
# Cell 12 — Paper figures
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'utils/paper_figures.py', '--n-samples', '4', '--n-eval', '200'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Figures] exit code', proc.returncode)
figs = os.path.join('results', 'figures')
if os.path.isdir(figs):
    for f in sorted(os.listdir(figs)):
        if f != '.gitkeep':
            print(' ', f, f'{os.path.getsize(os.path.join(figs, f))/1024:.0f} KB')

## Cell 13 — Persist results + checkpoints (run at the END of every session)

`/kaggle/working` is wiped between sessions, so save `results/` (CSVs + checkpoints) to a Kaggle dataset. Next session, attach it and Cell 5b restores everything.

**First time only** — create the dataset once (uncomment):
```python
# import json, os
# os.makedirs('/kaggle/working/ADVC/results', exist_ok=True)
# meta = {"title": "advc-results", "id": "<your-kaggle-username>/advc-results",
#         "licenses": [{"name": "CC0-1.0"}]}
# json.dump(meta, open('/kaggle/working/ADVC/results/dataset-metadata.json', 'w'))
# import subprocess; subprocess.run(['kaggle','datasets','create','-p','/kaggle/working/ADVC/results','-r','zip'], check=True)
```

In [ ]:
# Cell 13 — Save a new version of the results dataset (after the first-time create above)
import subprocess
subprocess.run(['kaggle', 'datasets', 'version',
                '-p', '/kaggle/working/ADVC/results',
                '-m', 'session results update', '-r', 'zip'], check=True)
print('Saved. Attach this dataset next session so Cell 5b can restore it.')

---
## Running across multiple sessions with your PC off

The full 36-row matrix (~20 GPU-h) does **not** fit in one 12h Kaggle session. Split it:

| Session | Cells to run | Approx |
|---|---|---|
| 1 | 1–7 (setup + Phase 1) then Cell 8 with `LEVELS=['fp32','int8']` | ~6h |
| 2 | 1–5c (restore) then Cell 8 with `LEVELS=['int4']` | ~3h |
| 3 | 1–5c then Cell 9 with `LEVELS=['fp32','int8']` | ~7h |
| 4 | 1–5c then Cell 9 with `LEVELS=['int4']` | ~4h |
| 5 | 1–5c then Cell 10 (Phase 3) + Cell 12 (figures) | ~4h |

**Every session ends with Cell 13** to persist results, and **every session from #2 starts by attaching your advc-results dataset** so Cell 5b + 5c restore prior work. Cells 8/9 auto-detect restored checkpoints and skip retraining.

### Interactive vs. Save & Run All
- **Interactive** (press ▶): recommended for the multi-session split — you control which levels run and call Cell 13 yourself.
- **Save & Run All (Commit)**: runs headless with your PC off, but re-runs the whole notebook from scratch on a fresh machine and **commits nothing if it times out**. Only use it for a scope that fits in 12h (e.g. one phase), and make sure Cell 5b restores prior checkpoints so it doesn't retrain finished levels.

### How to confirm it is NOT restarting from scratch
Run **Cell 5c** after restore. It prints, per level, whether a reusable `epoch07` checkpoint exists and which CSV rows are already recorded. In the training-cell output, a resumed level shows `... loading checkpoint ...` / `... checkpoint loaded.` with **no** `Epoch 1/7` lines; a resumed row shows `Resuming — N combination(s) already done`.